# Notebook 1 — EfficientNet-B4 Classification (Supervised CNN Baseline)

Fine-tune a pretrained EfficientNet-B4 on iWildCam species labels.
This serves as the CNN baseline for comparison.

**Colab GPU:** T4 (16 GB) is sufficient. Runtime → Change runtime type → T4 GPU.

In [ ]:
#@title 1. Setup
from pathlib import Path
import sys

repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/wild_animal_image_classification'),
    Path('/content'),
]
repo_root = None
for candidate in repo_candidates:
    if (candidate / 'configs' / 'config.py').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
import timm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torch.cuda.amp import GradScaler, autocast

from configs.config import CFG
from utils.dataset import make_loaders

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title 2. Load Pre-Saved Splits
CFG.ensure_output_dirs()
SPLITS_DIR = CFG.splits_dir
RESULTS_DIR = CFG.results_dir
CKPT_DIR = CFG.checkpoints_dir

train_df = pd.read_csv(f'{SPLITS_DIR}/train.csv')
val_df = pd.read_csv(f'{SPLITS_DIR}/val.csv')
test_df = pd.read_csv(f'{SPLITS_DIR}/test.csv')

with open(f'{SPLITS_DIR}/metadata.json') as f:
    meta = json.load(f)
NUM_CLASSES = meta['num_classes']

with open(f'{SPLITS_DIR}/cat_names.json') as f:
    cat_names = json.load(f)
with open(f'{SPLITS_DIR}/label_to_cat.json') as f:
    label_to_cat = json.load(f)

class_names = [cat_names.get(str(label_to_cat[str(i)]), f'class_{i}') for i in range(NUM_CLASSES)]

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Classes: {NUM_CLASSES}')

In [ ]:
#@title 3. Dataset and DataLoaders
IMG_SIZE = CFG.img_size_efficientnet
BATCH_SIZE = CFG.eff_batch_size
NUM_WORKERS = CFG.num_workers

train_loader, val_loader, test_loader = make_loaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=CFG.pin_memory,
)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
#@title 4. Build EfficientNet-B4 Model
import timm

model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'EfficientNet-B4: {total_params/1e6:.1f}M params ({trainable_params/1e6:.1f}M trainable)')

In [ ]:
#@title 5. Training Setup
# --- Hyperparameters (experiment with these!) ---
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 15
GRAD_ACCUM = 2  # effective batch = 24 * 2 = 48
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

CKPT_PATH = f'{CKPT_DIR}/efficientnet_b4_best.pth'
print(f'Checkpoint will be saved to: {CKPT_PATH}')

In [ ]:
#@title 6. Training Loop
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_f1 = 0.0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (i + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM

    train_loss = running_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_preds, val_labels_all = [], []
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item()
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_labels_all.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels_all, val_preds)
    val_f1 = f1_score(val_labels_all, val_preds, average='macro', zero_division=0)

    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Epoch {epoch:>2}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | {elapsed:.1f}s')

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'  -> New best F1: {best_f1:.4f} — checkpoint saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest Val F1: {best_f1:.4f}')

In [ ]:
#@title 7. Plot Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['val_acc'], 'g-', label='Val Accuracy')
ax2.plot(epochs_range, history['val_f1'], 'm-', label='Val F1 (macro)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.set_title('Validation Metrics'); ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('EfficientNet-B4 Training', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/efficientnet_b4_training_curves.png', dpi=150)
plt.show()

In [ ]:
#@title 8. Final Test Evaluation
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

test_preds, test_labels_all = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast():
            outputs = model(images)
        test_preds.extend(outputs.argmax(1).cpu().numpy())
        test_labels_all.extend(labels.numpy())

test_preds = np.array(test_preds)
test_labels_all = np.array(test_labels_all)

test_acc = accuracy_score(test_labels_all, test_preds)
test_f1 = f1_score(test_labels_all, test_preds, average='macro', zero_division=0)
test_f1_w = f1_score(test_labels_all, test_preds, average='weighted', zero_division=0)

print(f'\n{"="*50}')
print(f'  EfficientNet-B4 — TEST RESULTS')
print(f'{"="*50}')
print(f'  Accuracy:      {test_acc:.4f}')
print(f'  F1 (macro):    {test_f1:.4f}')
print(f'  F1 (weighted): {test_f1_w:.4f}')
print(f'{"="*50}\n')

print(classification_report(test_labels_all, test_preds, target_names=class_names, zero_division=0))

# Save report
report = classification_report(test_labels_all, test_preds, target_names=class_names,
                               output_dict=True, zero_division=0)
with open(f'{RESULTS_DIR}/efficientnet_b4_report.json', 'w') as f:
    json.dump(report, f, indent=2)

# Confusion matrix
cm = confusion_matrix(test_labels_all, test_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('EfficientNet-B4 — Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/efficientnet_b4_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
#@title 9. Save Final Results for Comparison Notebook
final_results = {
    'model': 'EfficientNet-B4',
    'accuracy': float(test_acc),
    'f1_macro': float(test_f1),
    'f1_weighted': float(test_f1_w),
    'best_val_f1': float(best_f1),
    'epochs_trained': len(history['train_loss']),
    'hyperparameters': {
        'lr': LR, 'weight_decay': WEIGHT_DECAY, 'batch_size': BATCH_SIZE,
        'img_size': IMG_SIZE, 'grad_accum': GRAD_ACCUM,
    }
}
with open(f'{RESULTS_DIR}/efficientnet_b4_final.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print('Results saved. Done!')